In [49]:
import duckdb
import pandas as pd

con = duckdb.connect("../notebooks/lakehouse.duckdb")
con.execute("SHOW TABLES").df()

,name
0,raw_customers
1,raw_orders
2,raw_products
3,stage_customers


In [50]:
con.execute("select * from raw_customers limit 10").df()

,customer_id,full_name,email,phone,city,age,registration_date,credit_card_last4
0,1.0,MARIA LOPEZ,maria1@gmail.com,+51 928728463,lima,22.0,2023-02-02,2679-9935-2424-7912
1,2.0,ana torres,ana2@gmail.com,None,Trujillo,19.0,04-04-2024,4257-9928-7873-4611
2,3.0,juan perez,juan3@gmail.com,+51 930868105,Trujillo,41.0,2024-01-25,4527-6514-2674-2519
3,4.0,SOFIA MENDOZA,sofia4@gmail.com,None,Trujillo,33.0,03-22-2023,8527-9785-3045-7201
4,5.0,Camila Vargas,camila5@gmail.com,+51 916150444,cusco,150.0,2025-06-06,4733-5741-2307-4814
5,6.0,Luis Garcia,luis6@gmail.com,+51 959684848,AREQUIPA,60.0,09-15-2024,6820-4432-5374-2169
6,7.0,Diego Fernandez,diego7@gmail.com,+51 995899313,Trujillo,19.0,2025-04-21,4598-6313-1916-4752
7,8.0,pedro castillo,None,+51 986125617,AREQUIPA,-5.0,20/10/2024,6155-4483-9179-7482
8,9.0,Luis Garcia,luis9@gmail.com,+51 945264581,cusco,41.0,05-22-2023,8019-7543-6930-4593
9,10.0,MARIA LOPEZ,maria10@gmail.com,+51 931472420,Piura,22.0,02/09/2024,7916-2040-7304-7252


In [51]:
#Traemos los datos crudos a una DataFrame de Pandas para su limpieza
df = con.execute("select * from raw_customers").df()

In [52]:
#Eliminamos filas donde todas las columnas son nulas
df = df.dropna(how="all")

In [53]:
#Eliminados registros con duplicados exactos
df = df.drop_duplicates()

In [54]:
# Convertimos texto de miniscula a mayusculas
#Las columnas son full_name y city

df["full_name"] = df["full_name"].str.upper()
df["city"] = df["city"].str.upper()

In [55]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 40 entries, 0 to 39
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        40 non-null     float64
 1   full_name          40 non-null     object 
 2   email              35 non-null     object 
 3   phone              32 non-null     object 
 4   city               38 non-null     object 
 5   age                36 non-null     float64
 6   registration_date  40 non-null     object 
 7   credit_card_last4  40 non-null     object 
dtypes: float64(2), object(6)
memory usage: 2.8+ KB


In [56]:
# Convertimos customer_id a numero entero
df["customer_id"] = df["customer_id"].astype("Int64")

In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 40 entries, 0 to 39
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        40 non-null     Int64  
 1   full_name          40 non-null     object 
 2   email              35 non-null     object 
 3   phone              32 non-null     object 
 4   city               38 non-null     object 
 5   age                36 non-null     float64
 6   registration_date  40 non-null     object 
 7   credit_card_last4  40 non-null     object 
dtypes: Int64(1), float64(1), object(6)
memory usage: 2.9+ KB


In [58]:
df.head()

,customer_id,full_name,email,phone,city,age,registration_date,credit_card_last4
0,1,MARIA LOPEZ,maria1@gmail.com,+51 928728463,LIMA,22.0,2023-02-02,2679-9935-2424-7912
1,2,ANA TORRES,ana2@gmail.com,None,TRUJILLO,19.0,04-04-2024,4257-9928-7873-4611
2,3,JUAN PEREZ,juan3@gmail.com,+51 930868105,TRUJILLO,41.0,2024-01-25,4527-6514-2674-2519
3,4,SOFIA MENDOZA,sofia4@gmail.com,None,TRUJILLO,33.0,03-22-2023,8527-9785-3045-7201
4,5,CAMILA VARGAS,camila5@gmail.com,+51 916150444,CUSCO,150.0,2025-06-06,4733-5741-2307-4814


In [59]:
# Validamos edades
df.loc[(df["age"] < 0) | (df["age"] > 120), "age"] = pd.NA

In [60]:
# En vez de dejar que pandas "adivine" un solo formato para toda la columna
# (lo que causaba que perdiera fechas válidas en otros formatos),
# probamos varios formatos conocidos, uno por uno, para cada valor.
def parse_fecha_flexible(valor):
    if pd.isna(valor):
        return pd.NaT
    formatos_posibles = ["%Y-%m-%d", "%d/%m/%Y", "%m-%d-%Y"]
    for fmt in formatos_posibles:
        try:
            return pd.to_datetime(valor, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT  # si NINGÚN formato conocido funcionó, ahí sí es irrecuperable

df["registration_date"] = df["registration_date"].apply(parse_fecha_flexible)

In [61]:
# Agregamos una columna de auditoria
df["updated_at"] = pd.Timestamp.now()

In [62]:
df.head()

,customer_id,full_name,email,phone,city,age,registration_date,credit_card_last4,updated_at
0,1,MARIA LOPEZ,maria1@gmail.com,+51 928728463,LIMA,22.0,2023-02-02,2679-9935-2424-7912,2026-06-16 17:56:43.706511
1,2,ANA TORRES,ana2@gmail.com,None,TRUJILLO,19.0,2024-04-04,4257-9928-7873-4611,2026-06-16 17:56:43.706511
2,3,JUAN PEREZ,juan3@gmail.com,+51 930868105,TRUJILLO,41.0,2024-01-25,4527-6514-2674-2519,2026-06-16 17:56:43.706511
3,4,SOFIA MENDOZA,sofia4@gmail.com,None,TRUJILLO,33.0,2023-03-22,8527-9785-3045-7201,2026-06-16 17:56:43.706511
4,5,CAMILA VARGAS,camila5@gmail.com,+51 916150444,CUSCO,NaN,2025-06-06,4733-5741-2307-4814,2026-06-16 17:56:43.706511


In [63]:
# convertimos age a entero
df["age"] = df["age"].astype("Int64")

In [64]:
#Guardamos el raw en tablas stage
con.execute("CREATE OR REPLACE TABLE stage_customers AS SELECT * FROM df")

In [65]:
con.execute("SHOW TABLES").df()


,name
0,raw_customers
1,raw_orders
2,raw_products
3,stage_customers


In [66]:

con.execute("SELECT * FROM stage_customers limit 3").df()

,customer_id,full_name,email,phone,city,age,registration_date,credit_card_last4,updated_at
0,1,MARIA LOPEZ,maria1@gmail.com,+51 928728463,LIMA,22,2023-02-02,2679-9935-2424-7912,2026-06-16 17:56:43.706511
1,2,ANA TORRES,ana2@gmail.com,None,TRUJILLO,19,2024-04-04,4257-9928-7873-4611,2026-06-16 17:56:43.706511
2,3,JUAN PEREZ,juan3@gmail.com,+51 930868105,TRUJILLO,41,2024-01-25,4527-6514-2674-2519,2026-06-16 17:56:43.706511


In [67]:
con.execute("SELECT * FROM raw_customers limit 3").df()

,customer_id,full_name,email,phone,city,age,registration_date,credit_card_last4
0,1.0,MARIA LOPEZ,maria1@gmail.com,+51 928728463,lima,22.0,2023-02-02,2679-9935-2424-7912
1,2.0,ana torres,ana2@gmail.com,None,Trujillo,19.0,04-04-2024,4257-9928-7873-4611
2,3.0,juan perez,juan3@gmail.com,+51 930868105,Trujillo,41.0,2024-01-25,4527-6514-2674-2519


In [69]:
con.execute("SELECT * FROM raw_orders limit 3").df()

,order_id,customer_id,product_id,quantity,order_date,total_amount_usd,status
0,1,18,11,3,2025-04-02,49.5,pending
1,2,15,13,5,2024-03-19,19.9,Completed
2,3,27,11,-1,26/08/2024,19.9,pending


In [70]:
# Limpiamos la tabla de orders
df_orders = con.execute("SELECT * FROM raw_orders").df()

df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          81 non-null     int64  
 1   customer_id       81 non-null     int64  
 2   product_id        81 non-null     int64  
 3   quantity          81 non-null     int64  
 4   order_date        81 non-null     object 
 5   total_amount_usd  81 non-null     float64
 6   status            81 non-null     object 
dtypes: float64(1), int64(4), object(2)
memory usage: 4.6+ KB


In [74]:


# Eliminamos filas nulas y con duplicados
df_orders = df_orders.dropna(how="all")
df_orders = df_orders.drop_duplicates()

# Convertimos status a mayusculas
df_orders["status"] = df_orders["status"].str.upper()

# Validamos Quantity
df_orders = df_orders[df_orders["quantity"] > 0]

# Convertimos fecha a un formato de fechas con la funcion previamente hecha
df_orders["order_date"] = df_orders["order_date"].apply(parse_fecha_flexible)

# Con df.info() verificamos que los id son enteros
# Creamos la columna de auditoria
df_orders["updated_at"] = pd.Timestamp.now()

df_orders.head()

,order_id,customer_id,product_id,quantity,order_date,total_amount_usd,status,updated_at
0,1,18,11,3,2025-04-02,49.5,PENDING,2026-06-16 18:05:40.047974
1,2,15,13,5,2024-03-19,19.9,COMPLETED,2026-06-16 18:05:40.047974
4,5,31,1,3,2024-06-01,500.0,PENDING,2026-06-16 18:05:40.047974
5,6,35,20,2,2024-09-06,99.0,COMPLETED,2026-06-16 18:05:40.047974
8,9,37,1,1,2024-11-25,49.5,COMPLETED,2026-06-16 18:05:40.047974


In [72]:
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
Index: 54 entries, 0 to 79
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          54 non-null     int64         
 1   customer_id       54 non-null     int64         
 2   product_id        54 non-null     int64         
 3   quantity          54 non-null     int64         
 4   order_date        54 non-null     datetime64[ns]
 5   total_amount_usd  54 non-null     float64       
 6   status            54 non-null     object        
 7   updated_at        54 non-null     datetime64[us]
dtypes: datetime64[ns](1), datetime64[us](1), float64(1), int64(4), object(1)
memory usage: 3.8+ KB


In [75]:
# Guardamos la tabla limpia de orders en stage
con.execute("CREATE OR REPLACE TABLE stage_orders AS SELECT * FROM df_orders")

In [76]:
con.execute("SHOW TABLES").df()

,name
0,raw_customers
1,raw_orders
2,raw_products
3,stage_customers
4,stage_orders


In [81]:
con.execute("SELECT * FROM raw_products limit 5").df()

,product_id,product_name,category,price,discount_pct,stock
0,1,Producto 1,deportes,5.0,20,50.0
1,2,Producto 2,ROPA,-10.0,0,NaN
2,3,Producto 3,ROPA,-10.0,0,10.0
3,4,Producto 4,Belleza,120.0,120,-3.0
4,5,Producto 5,Hogar,19.9,10,0.0


In [77]:
# Limpiamos la tabla de products
df_products = con.execute("SELECT * FROM raw_products").df()

df_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    21 non-null     int64  
 1   product_name  21 non-null     object 
 2   category      21 non-null     object 
 3   price         21 non-null     float64
 4   discount_pct  21 non-null     int64  
 5   stock         16 non-null     float64
dtypes: float64(2), int64(2), object(2)
memory usage: 1.1+ KB


In [82]:
# Eliminamos filas con valores nulos y duplicados
df_products = df_products.dropna(how="all")
df_products = df_products.drop_duplicates()

#Convertimos product_name y category a mayusculas
df_products["product_name"] = df_products["product_name"].str.upper()
df_products["category"] = df_products["category"].str.upper()

# Validamos Price
df_products = df_products[df_products["price"] >= 0]

# Validamos descuento, no puede ser negativo ni mayor al 100%
df_products.loc[df_products["discount_pct"]>100, "discount_pct"] = pd.NA

# Validamos que no pueda haber stock negativo y convertimos a entero
df_products.loc[df_products["stock"] < 0, "stock"] = pd.NA
df_products["stock"] = df_products["stock"].astype("Int64")

# Agregamos columna de auditoria
df_products["update_at"]= pd.Timestamp.now()

df_products.head()

,product_id,product_name,category,price,discount_pct,stock,update_at
0,1,PRODUCTO 1,DEPORTES,5.0,20.0,50,2026-06-16 18:15:52.682093
3,4,PRODUCTO 4,BELLEZA,120.0,NaN,<NA>,2026-06-16 18:15:52.682093
4,5,PRODUCTO 5,HOGAR,19.9,10.0,0,2026-06-16 18:15:52.682093
5,6,PRODUCTO 6,ELECTRONICA,49.5,10.0,10,2026-06-16 18:15:52.682093
6,7,PRODUCTO 7,BELLEZA,49.5,50.0,10,2026-06-16 18:15:52.682093


In [83]:
df_products.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13 entries, 0 to 17
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   product_id    13 non-null     int64         
 1   product_name  13 non-null     object        
 2   category      13 non-null     object        
 3   price         13 non-null     float64       
 4   discount_pct  10 non-null     float64       
 5   stock         8 non-null      Int64         
 6   update_at     13 non-null     datetime64[us]
dtypes: Int64(1), datetime64[us](1), float64(2), int64(1), object(2)
memory usage: 845.0+ bytes


In [84]:
# Guardamos la tabla en stage
con.execute("CREATE OR REPLACE TABLE stage_products AS SELECT * FROM df_products")

In [91]:
con.execute("SHOW TABLES").df()

ConnectionException: Connection Error: Connection already closed!

In [90]:
con.close()